In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# 加载加州房价数据集
# MedInc: median income in block group
# HouseAge: median house age in block group
# AveRooms: average number of rooms per household
# AveBedrms: average number of bedrooms per household
# Population: block group population
# AveOccup: average number of household members
# Latitude: block group latitude
# Longitude: block group longitude
california_housing = fetch_california_housing()
print(california_housing)
# 将数据转换为 Pandas DataFrame
data = pd.DataFrame(california_housing.data, columns=california_housing.feature_names)
data['MedHouseVal'] = california_housing.target

# 显示数据集的前几行
print(data.head())


{'data': array([[   8.3252    ,   41.        ,    6.98412698, ...,    2.55555556,
          37.88      , -122.23      ],
       [   8.3014    ,   21.        ,    6.23813708, ...,    2.10984183,
          37.86      , -122.22      ],
       [   7.2574    ,   52.        ,    8.28813559, ...,    2.80225989,
          37.85      , -122.24      ],
       ...,
       [   1.7       ,   17.        ,    5.20554273, ...,    2.3256351 ,
          39.43      , -121.22      ],
       [   1.8672    ,   18.        ,    5.32951289, ...,    2.12320917,
          39.43      , -121.32      ],
       [   2.3886    ,   16.        ,    5.25471698, ...,    2.61698113,
          39.37      , -121.24      ]]), 'target': array([4.526, 3.585, 3.521, ..., 0.923, 0.847, 0.894]), 'frame': None, 'target_names': ['MedHouseVal'], 'feature_names': ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude'], 'DESCR': '.. _california_housing_dataset:\n\nCalifornia Housing dataset\n-

In [8]:
# 拆分数据集
X = data.drop('MedHouseVal', axis=1)
y = data['MedHouseVal']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# 标准化数据
# 为什么需要 StandardScaler
#  - 提升模型性能：许多机器学习算法（如线性回归、支持向量机、K近邻算法、神经网络等）在不同特征尺度下表现不佳。标准化可以使这些算法在训练时更快地收敛，并且可能提高模型的预测性能。
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
# transform 方法仅对数据进行转换，而不计算新的统计信息。它使用 fit 方法计算并存储的统计信息对数据进行转换。通常在对测试数据或新数据进行预处理时使用，以确保训练和测试数据使用相同的转换规则。
X_test = scaler.transform(X_test)


[[-1.15508475 -0.28632369 -0.52068576 ...  0.06740798  0.1951
   0.28534728]
 [-0.70865905  0.11043502 -0.16581537 ... -0.03602975 -0.23549054
   0.06097472]
 [-0.21040155  1.85617335 -0.61076476 ... -0.14998876  1.00947776
  -1.42487026]
 ...
 [ 2.80902421 -0.28632369  0.75501156 ... -0.02646898  0.78014149
  -1.23041404]
 [-0.57542978  0.58654547 -0.06124296 ... -0.04390537  0.52740357
  -0.08860699]
 [-0.17259111 -0.92113763 -0.6058703  ...  0.05466644 -0.66608108
   0.60445493]]


In [3]:
# 构建模型
model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.2),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(1)  # 输出层，1个神经元用于回归
])

# 编译模型
model.compile(optimizer='adam', loss='mse', metrics=['mae'])


/Users/huke/openai-learning/venv2405/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [4]:
# 训练模型
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2)

# 评估模型
test_loss, test_mae = model.evaluate(X_test, y_test)
# test_mae 表示在测试数据集上计算的平均绝对误差（Mean Absolute Error，MAE）。MAE 是一种衡量模型预测值与真实值之间平均误差的指标，它通过计算预测值和真实值之间绝对误差的平均值来衡量模型的性能。
# Test MAE: 0.3592793047428131
# 如果数据集中的房价单位是数千美元，那么这个误差意味着模型预测的房价与真实房价之间的平均误差约为 359 美元。
print(f"Test MAE: {test_mae}")


Epoch 1/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 1s 964us/step - loss: 1.5541 - mae: 0.8906 - val_loss: 0.5017 - val_mae: 0.4861
Epoch 2/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 435us/step - loss: 0.5696 - mae: 0.5481 - val_loss: 0.4296 - val_mae: 0.4616
Epoch 3/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 456us/step - loss: 0.5237 - mae: 0.5202 - val_loss: 0.4131 - val_mae: 0.4501
Epoch 4/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 423us/step - loss: 0.4646 - mae: 0.4909 - val_loss: 0.4006 - val_mae: 0.4451
Epoch 5/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 415us/step - loss: 0.4557 - mae: 0.4801 - val_loss: 0.4269 - val_mae: 0.4430
Epoch 6/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 422us/step - loss: 0.4199 - mae: 0.4635 - val_loss: 0.3863 - val_mae: 0.4329
Epoch 7/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 416us/step - loss: 0.4200 - mae: 0.4661 - val_loss: 0.3787 - val_mae: 0.4364
Epoch 8/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 0s 415us/step - loss: 0.4047 - mae: 0.4507 - val_loss: 0.3740 - val_mae: 0.4264
Epoch 9/50
413/413 ━━━━━━━━━━━━━━━━━━━━ 